# Global Conv1D Attention Challenger

Reproduces the evaluator architecture with causal convolutions, self-attention, spectral and known-future branches, ordered quantiles, five deterministic seeds and a challenger-only promotion gate.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd()
if not (ROOT / "outputs").exists():
    ROOT = Path("Ai miroservices/modeling/project_operational_baseline").resolve()
OUT = ROOT / "outputs"
EVAL = OUT / "evaluator"
sns.set_theme(style="whitegrid")

## Architecture and claim boundary

- Input: 24 historical months; direct output: H1-H12.
- Two causal Conv1D layers use 3- and 6-month kernels.
- Two residual four-head self-attention blocks learn non-local dependence.
- Calendar, spectral, static and origin-known future features join after temporal pooling.
- P10/P50/P90 and cost-sensitive quantiles are ordered by construction.

Attention is descriptive, not a causal explanation. The network becomes champion only if locked pre-test evidence supports it.

In [ ]:
summary=json.loads((EVAL/'evaluator_run_summary.json').read_text())
seed=pd.read_csv(EVAL/'neural_seed_stability.csv')
ablations=pd.read_csv(EVAL/'feature_group_ablations.csv')
display(pd.DataFrame([summary]).T)
display(seed.groupby('origin_month')[['WAPE','RMSE','epochs']].agg(['mean','std','min','max']))
display(ablations.sort_values('WAPE'))

In [ ]:
attention=pd.read_csv(EVAL/'attention_weights.csv')
occlusion=pd.read_csv(EVAL/'lag_occlusion_sensitivity.csv')
permutation=pd.read_csv(EVAL/'heldout_group_permutation.csv')
fig,axes=plt.subplots(1,2,figsize=(14,5))
sns.lineplot(data=attention,x='lag_position',y='mean_attention_weight',marker='o',ax=axes[0])
sns.barplot(data=occlusion.sort_values('WAPE_increase',ascending=False).head(12),x='WAPE_increase',y='lag_position',orient='h',ax=axes[1]); plt.tight_layout(); plt.show()
display(permutation.sort_values('WAPE_increase',ascending=False))

In [ ]:
history=pd.read_csv(EVAL/'neural_training_history.csv')
sns.lineplot(data=history,x='epoch',y='validation_loss',hue='origin_month',legend=False); plt.title('Validation loss by rolling origin and seed'); plt.show()